In [ ]:
import requests
import pandas as pd

END = pd.Timestamp.today().normalize()

# Past ECMWF runs at 0.25° are archived from 15 Feb 2024.
START = pd.Timestamp("2024-02-15")
BASE = 18.0
LEADS = range(1, 8)  # days ahead

# weight = Eurostat 2023 inland consumption of natural gas, bcm
CITIES = {
    "DE": (52.52, 13.41, 79.82),  # Berlin
    "IT": (45.46, 9.19, 61.74),   # Milan
    "FR": (48.85, 2.35, 33.36),   # Paris
    "ES": (40.42, -3.70, 28.96),  # Madrid
    "NL": (52.37, 4.90, 31.40),   # Amsterdam
    "PL": (52.23, 21.01, 20.22),  # Warsaw
    "BE": (50.85, 4.35, 14.61),   # Brussels
    "RO": (44.43, 26.10, 9.58),   # Bucharest
    "HU": (47.50, 19.04, 8.65),   # Budapest
    "AT": (48.21, 16.37, 7.02),   # Vienna
}

In [2]:
def pull(lead):
    # previous_day{k} on date D is the temperature ECMWF forecast for D, issued k days earlier.
    r = requests.get(
        "https://previous-runs-api.open-meteo.com/v1/forecast",
        params={
            "latitude": ",".join(str(lat) for lat, _, _ in CITIES.values()),
            "longitude": ",".join(str(lon) for _, lon, _ in CITIES.values()),
            "hourly": f"temperature_2m_previous_day{lead}",
            "models": "ecmwf_ifs025",
            "start_date": START.strftime("%Y-%m-%d"),
            "end_date": END.strftime("%Y-%m-%d"),
            "timezone": "Europe/Berlin",
        },
        timeout=180,
    )
    print(lead, r.status_code, len(r.content))
    r.raise_for_status()
    data = r.json()
    if isinstance(data, dict):
        data = [data]
    col = f"temperature_2m_previous_day{lead}"
    daily = {}
    for city, loc in zip(CITIES, data):
        s = pd.Series(loc["hourly"][col], index=pd.to_datetime(loc["hourly"]["time"]))
        daily[city] = s.resample("D").mean()
    out = pd.DataFrame(daily)
    out.index = out.index.tz_localize(None)
    out.index.name = "date"
    return out

valid = {lead: pull(lead) for lead in LEADS}
valid[1].tail()

1 200 5523435
2 200 5523452
3 200 5523628
4 200 5523493
5 200 5523524
6 200 5522943
7 200 5521963


,DE,IT,FR,ES,NL,PL,BE,RO,HU,AT
date,,,,,,,,,,
2026-10-08,13.141667,16.933333,12.583333,14.937500,13.062500,15.170833,12.829167,17.058333,17.816667,15.662500
2026-10-09,10.529167,16.795833,12.862500,12.562500,12.812500,11.850000,12.408333,18.804167,15.458333,13.691667
2026-10-10,13.062500,16.375000,14.062500,15.179167,11.587500,12.500000,12.125000,18.554167,14.266667,14.904167
2026-10-11,9.666667,15.845833,13.037500,16.866667,10.750000,11.591667,10.504167,19.487500,13.825000,12.791667
2026-10-12,9.229167,15.658333,13.275000,18.879167,10.570833,7.766667,11.691667,14.841667,12.120833,11.591667


In [3]:
valid[1].head()

,DE,IT,FR,ES,NL,PL,BE,RO,HU,AT
date,,,,,,,,,,
2024-02-15,9.975000,6.687500,11.758333,11.387500,11.866667,5.741667,12.625000,6.129167,6.604167,9.008333
2024-02-16,11.104167,7.533333,12.208333,10.479167,10.400000,6.641667,12.220833,6.316667,6.208333,7.595833
2024-02-17,9.158333,8.512500,9.141667,8.154167,8.704167,7.587500,9.750000,5.079167,5.604167,8.416667
2024-02-18,5.341667,8.562500,8.979167,7.779167,9.162500,2.075000,9.833333,5.995833,6.308333,7.033333
2024-02-19,7.912500,9.029167,10.662500,9.725000,7.858333,1.741667,8.641667,5.241667,4.708333,7.833333


In [ ]:
# One variable: how much colder or warmer ECMWF said the next 7 days would be,
# versus the previous 10 years of realised EU heating degree-days on the same calendar day.

weights = pd.Series({city: CITIES[city][2] for city in CITIES}, dtype=float)
weights = weights / weights.sum()
days = pd.date_range(START, END, freq="D")

def eu_hdd(temp):
    hdd = (BASE - temp).clip(lower=0)
    ok = hdd.notna().all(axis=1)
    return hdd.mul(weights, axis=1).sum(axis=1).where(ok)

issued = []
for lead, temp in valid.items():
    # Stored on the valid date. Move it back onto the issue date.
    issued.append(eu_hdd(temp).reindex(days).shift(-lead).rename(lead))
fc = pd.concat(issued, axis=1)

hist = pd.read_csv("weather_hist.csv", parse_dates=["date"], index_col="date")["EU_HDD"]

def normal_on(dates):
    # Same month-day in the previous 10 years. The year of the valid date is excluded.
    vals = []
    for date in dates:
        month, day = date.month, (28 if date.month == 2 and date.day == 29 else date.day)
        keys = pd.to_datetime([f"{year}-{month:02d}-{day:02d}" for year in range(date.year - 10, date.year)])
        vals.append(hist.reindex(keys).mean())
    return pd.Series(vals, index=fc.index)

parts = []
for lead in LEADS:
    parts.append(fc[lead] - normal_on(fc.index + pd.Timedelta(days=lead)))
out = pd.DataFrame({"fc_hdd_anom7": pd.concat(parts, axis=1).sum(axis=1, min_count=7)})
out = out.dropna()
out.index.name = "date"
out.to_csv("weather_forecast.csv", float_format="%.2f")
out.tail()

,fc_hdd_anom7
date,
2026-10-01,-16.895097
2026-10-02,-12.131436
2026-10-03,-10.814097
2026-10-04,-6.550678
2026-10-05,-6.378374
